# 5.1 Course Bottleneck Detection with Unsupervised Learning

## Course 3: Advanced Classification Models for Student Success

## Opening Narrative

> *"We've spent this course predicting who will leave. Now we ask a different question:
> **Which courses are quietly working against our students?**"*

### The Provost's Question

Imagine your Provost says:

> *"Some courses seem to act as **bottlenecks** — students hit them, fail or withdraw,
> repeat them a semester or two later, and lose momentum toward their degree.
> We have data on 1,000 course sections. Can we **discover** which courses are the
> real bottlenecks — and rank them so the Curriculum Committee knows where to start?"*

**That question is the heart of unsupervised learning.**

Unlike supervised learning (Modules 1–4), where we predicted a known target (`DEPARTED`),
unsupervised learning has **no target variable**. No one has labeled which courses are
"bottlenecks." Instead, we let the data reveal its own structure — clusters of similar
course sections — and then translate those patterns into an action plan.

### What You Will Learn

| Concept | What It Does | Institutional Use |
|:--------|:-------------|:------------------|
| **K-Means Clustering** | Groups similar observations into k clusters | Segment course sections into distinct profiles |
| **PCA** (Principal Component Analysis) | Reduces many variables to a few key dimensions | Visualize 13-variable course data in 2D |
| **Elbow Method & Silhouette Score** | Determine optimal number of clusters | Justify the number of course profiles to leadership |
| **Cluster Profiling** | Describe what makes each cluster unique | Translate patterns into curriculum-reform priorities |
| **Risk Prioritization Index** | Combine metrics into one ranked score | Hand leadership a ranked list of courses to review |

### Key Difference from Supervised Learning

| | Supervised (Modules 1–4) | Unsupervised (This Module) |
|:--|:------------------------|:--------------------------|
| **Goal** | Predict a known outcome | Discover hidden structure |
| **Target variable** | Yes (`DEPARTED`) | No |
| **Evaluation** | AUC, F1, Accuracy | Silhouette, Inertia, Interpretability |
| **Question** | "Will this student leave?" | "What *kinds* of courses do we have?" |
| **Action** | Flag individual students | Prioritize course-level reform |

## 1. Setup and Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Visualization defaults
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

print("All libraries loaded successfully!")
print("This module uses matplotlib/seaborn for static, publication-ready visuals.")

## 2. Load the Course Section Data

The dataset lives in Google Drive alongside the other Course 3 data. We mount Drive,
point at the `data/` folder, and read `bottleneck.csv`.

> **Note:** In earlier modules we *generated* synthetic data inline. Here the synthetic
> course-section data has been pre-built and saved to `bottleneck.csv` so every run of
> this notebook analyzes exactly the same 1,000 sections.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set up file paths
project_path = '/content/drive/MyDrive/Applied-Data-Analytics-For-Higher-Education-Course-3'
data_filepath = '/data/'

# Load the course section data
course_df = pd.read_csv(f'{project_path}{data_filepath}bottleneck.csv')

print(f"Dataset: {course_df.shape[0]:,} course sections × {course_df.shape[1]} variables")
course_df.head()

### Variables (13)

| Variable | Description |
|:---------|:------------|
| `ENROLLMENT` | Section enrollment |
| `DFW_RATE` | DFW rate for the section (D, F, or Withdraw) |
| `PASS_RATE` | Pass rate (C or better) |
| `AVG_GPA` | Average grade in the section |
| `REPEAT_RATE` | Proportion of students repeating |
| `AVG_REPEAT_DELAY` | Average semesters before repeat |
| `SECTION_SIZE` | Number of seats |
| `PCT_FIRST_YEAR` | Proportion of first-year students |
| `PCT_STEM` | Proportion of STEM majors |
| `INSTRUCTOR_RATING` | Student evaluation score (1–5) |
| `PREREQUISITE_COUNT` | Number of prerequisites |
| `IS_GATEWAY` | Gateway course flag (0/1) |
| `UNITS` | Course unit value |

In [ ]:
# Quick summary of the raw variables
course_df.describe().round(2)

## 3. The Clustering Pipeline

We follow the same five-step pipeline used throughout this module:

**Scale → Choose k → Fit K-Means → Visualize with PCA → Profile the clusters.**

### Step 1: Scale the Data

K-Means measures distance, so variables on large scales (like `ENROLLMENT`, in the
hundreds) would dominate variables on small scales (like `DFW_RATE`, between 0 and 1).
`StandardScaler` puts every variable on a comparable footing (mean 0, standard deviation 1).

In [ ]:
scaler = StandardScaler()
course_scaled = scaler.fit_transform(course_df)

print(f"Scaled matrix shape: {course_scaled.shape}")
print(f"Mean of each scaled column (~0): {course_scaled.mean(axis=0).round(2)[:5]} ...")
print(f"Std of each scaled column  (~1): {course_scaled.std(axis=0).round(2)[:5]} ...")

### Step 2: Find the Number of Clusters

We sweep `k` from 2 to 10 and compute the **silhouette score** at each value
(higher = better-separated clusters). We use this, together with what the clusters
*mean*, to settle on a number of course profiles we can defend to leadership.

For this analysis we choose **k = 4**: four course profiles is both interpretable for
the Curriculum Committee and clearly recovers the distinct bottleneck group we care about.

In [ ]:
# Silhouette analysis across candidate k values
sils_c = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    sils_c.append(silhouette_score(course_scaled, km.fit_predict(course_scaled)))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(2, 11), sils_c, 'go-', linewidth=2, markersize=8)
ax.set_xlabel('k'); ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Analysis — Course Sections')
ax.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='k = 4 (chosen)')
ax.legend(); plt.tight_layout(); plt.show()

### Step 3: Fit K-Means with k = 4

In [ ]:
km_course = KMeans(n_clusters=4, n_init=10, random_state=RANDOM_STATE)
course_df['Cluster'] = km_course.fit_predict(course_scaled)

print("Sections per cluster:")
print(course_df['Cluster'].value_counts().sort_index())

### Step 4: PCA for Visualization

We can't plot 13 dimensions. **PCA** compresses the scaled features into two components
that capture as much variation as possible, so we can *see* whether the clusters
actually separate.

In [ ]:
pca_c = PCA(n_components=2, random_state=RANDOM_STATE)
course_pca = pca_c.fit_transform(course_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(course_pca[:, 0], course_pca[:, 1],
                     c=course_df['Cluster'], cmap='Set2', alpha=0.6, s=30,
                     edgecolors='w', linewidth=0.3)
ax.set_xlabel(f'PC1 ({pca_c.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca_c.explained_variance_ratio_[1]:.1%})')
ax.set_title('Course Section Segments (PCA)')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout(); plt.show()

### Step 5: Cluster Profiling

A cluster label is meaningless until we describe *what makes it distinct*. We compute
the mean of every variable within each cluster, then z-score the profile table so a
heatmap shows at a glance which clusters are concerning (red) and which are favorable (green).

In [ ]:
# Cluster profile: mean of each variable per cluster
profile_c = course_df.groupby('Cluster').mean().round(3)
profile_c['Count'] = course_df['Cluster'].value_counts().sort_index()
print("=" * 100)
print("CLUSTER PROFILES — Course Bottleneck Detection")
print("=" * 100)
print(profile_c.to_string())
print("=" * 100)

# Heatmap of z-scored profiles
profile_cz = (profile_c.drop(columns='Count') - profile_c.drop(columns='Count').mean()) / profile_c.drop(columns='Count').std()
fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(profile_cz, annot=True, fmt='.2f', cmap='RdYlGn_r', center=0, linewidths=1, ax=ax)
ax.set_title('Course Cluster Profiles (z-scored — red = concerning, green = favorable)')
ax.set_ylabel('Cluster')
plt.tight_layout(); plt.show()

### Reading the Profiles

Scan the heatmap for the cluster that is **red on the metrics that matter**: high
`DFW_RATE`, high `REPEAT_RATE`, low `PASS_RATE`, low `AVG_GPA`. That cluster is your
**bottleneck group** — typically large, gateway, STEM-heavy sections packed with
first-year students. The other clusters describe healthy core courses, high-demand
large lectures, and small upper-division sections.

Now we turn that qualitative read into a single ranked score.

## 4. Risk Prioritization Index

Clusters tell leadership *which kinds* of courses are problematic. To tell them *which
specific sections to review first*, we build a composite **Risk Prioritization Index**
for every section.

The index combines three weighted factors:
- **DFW Rate** (weight: 0.4) — direct measure of student failure
- **Repeat Rate** (weight: 0.3) — indicates students are stuck
- **Inverse Pass Rate** (weight: 0.3) — lower pass rate = higher risk

Each factor is normalized to 0–1 before weighting, so the final index is comparable across sections.

In [ ]:
# Risk Prioritization Index
course_df['RISK_INDEX'] = (
    0.4 * (course_df['DFW_RATE'] / course_df['DFW_RATE'].max()) +
    0.3 * (course_df['REPEAT_RATE'] / course_df['REPEAT_RATE'].max()) +
    0.3 * ((1 - course_df['PASS_RATE']) / (1 - course_df['PASS_RATE']).max())
).round(3)

# Top 20 highest-risk sections
top_risk = course_df.nlargest(20, 'RISK_INDEX')[
    ['ENROLLMENT', 'DFW_RATE', 'PASS_RATE', 'REPEAT_RATE', 'IS_GATEWAY', 'RISK_INDEX', 'Cluster']
]

print("=" * 80)
print("TOP 20 HIGHEST-RISK COURSE SECTIONS")
print("=" * 80)
print(top_risk.to_string())

# Risk distribution by cluster
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot
course_df.boxplot(column='RISK_INDEX', by='Cluster', ax=axes[0])
axes[0].set_title('Risk Index by Cluster')
axes[0].set_xlabel('Cluster'); axes[0].set_ylabel('Risk Index')
plt.sca(axes[0]); plt.title('Risk Index by Cluster')

# Histogram
for c in sorted(course_df['Cluster'].unique()):
    axes[1].hist(course_df[course_df['Cluster'] == c]['RISK_INDEX'],
                 alpha=0.5, bins=20, label=f'Cluster {c}')
axes[1].set_xlabel('Risk Index'); axes[1].set_ylabel('Count')
axes[1].set_title('Risk Index Distribution by Cluster')
axes[1].legend()
plt.tight_layout(); plt.show()

### Final Capstone Deliverables

In a real institutional analysis, you would deliver:

1. **Cluster summary table** — One row per cluster with mean statistics and interpretation
2. **PCA scatter plot** — Visual proof that clusters are separable
3. **Risk Prioritization Index** — Ranked list of courses needing review
4. **Executive memo** — 1-page summary for the Provost (see template below)

#### Executive Memo Template

> **To:** Provost / Curriculum Committee
>
> **From:** Institutional Research
>
> **Re:** Course Bottleneck Analysis — Risk-Prioritized Findings
>
> Using K-Means clustering on 13 course-level metrics for 1,000 sections,
> we identified four distinct course profiles. Cluster [X] contains the highest-risk
> sections (avg DFW rate = [Y]%, avg repeat rate = [Z]%). These [N] sections
> account for [P]% of total DFW enrollments despite being only [Q]% of sections.
>
> **Recommendation:** Prioritize curriculum review for the top 20 sections
> identified by the Risk Prioritization Index (attached).